In [5]:
from hcipy import *
import numpy as np
import matplotlib.pyplot as plt
from image_sharpening import FocusDiversePhaseRetrieval, ft_rev, mft_rev, InstrumentConfiguration
from skimage.transform import resize
import numpy.ma as ma
from skimage.restoration import unwrap_phase
from hcipy import *
import numpy as np
import matplotlib.pyplot as plt
from image_sharpening import FocusDiversePhaseRetrieval, ft_rev, mft_rev, InstrumentConfiguration
from skimage.transform import resize
from skimage.restoration import unwrap_phase


In [12]:
#Constants
pupil_size = 10.12e-3
small_pupil_size = 9.5e-3 
focal_length = 500e-3
wavelength = 650e-9
grid_size = 256
q = 16
num_airy = 16
f=focal_length
D=pupil_size
dx_list = [2.0071, 2.0071]
    # Create the pupil grid
pupil_grid = make_pupil_grid(256, pupil_size)#x could be grid_size
    
    # Create the focal grid
focal_grid = make_focal_grid(q=q, num_airy=num_airy, pupil_diameter=pupil_size, focal_length=focal_length, reference_wavelength=wavelength)
    
    # Aperture setup
aperture = circular_aperture(pupil_size)
telescope_pupil = aperture(pupil_grid)  # Define the telescope pupil
    
small_aperture = circular_aperture(small_pupil_size)
masking_pupil = small_aperture(pupil_grid)  # Define the masking pupil (if needed later)


/var/folders/fp/0vqz18r15679y3w674kfj3k40000gq/T/ipykernel_71552/747219922.py:19: DeprecationWarning: circular_aperture is deprecated. Its new name is make_circular_aperture.
  aperture = circular_aperture(pupil_size)
/var/folders/fp/0vqz18r15679y3w674kfj3k40000gq/T/ipykernel_71552/747219922.py:22: DeprecationWarning: circular_aperture is deprecated. Its new name is make_circular_aperture.
  small_aperture = circular_aperture(small_pupil_size)


In [18]:
def phase_to_m(phase, wv):
    return phase * wv / (2 * np.pi)

def p_to_delta(P, f, D):
    return 8 * P * (f/D)**2

def delta_to_p(delta, f, D):
    return -1 * delta / (8 * (f/D)**2)
    

def calculate_defocus_params(example_defocus, scale, f, D):
    """
    Calculates P2V error and defocus distance for given defocus phase and scale.

    Parameters
    ----------
    example_defocus : np.ndarray
        Array representing the defocus phase.
    scale : float
        Scale factor for the defocus phase.
    f : float
        Focal length of the system.
    D : float
        Size of the pupil.

    Returns
    -------
    tuple
        P2V error in radians and defocus distance.
    """

    defocus_phase = example_defocus * scale
    p2v_radians = np.max(defocus_phase) - np.min(defocus_phase)
    p2v_m = phase_to_m(p2v_radians, 650e-9)
    delta = p_to_delta(p2v_m, f, D)
    delta=delta if scale>0 else -1*delta
    return p2v_radians, delta

def propagate_image(defocus_phase, test_ab, telescope_pupil, wavelength):
    """
    Propagates an image using the given defocus phase, test aberration, and telescope pupil.

    Parameters
    ----------
    defocus_phase : np.ndarray
        Array representing the defocus phase.
    test_ab : np.ndarray
        Array representing the test aberration.
    telescope_pupil : np.ndarray
        Array representing the telescope pupil.
    wavelength : float
        Wavelength of light in meters.

    Returns
    -------
    focal_intensity
        Focal intensity of the propagated image.

    """    
    # Fraunhofer propagators
    prop_p2f = FraunhoferPropagator(pupil_grid, focal_grid, focal_length=focal_length)
    prop_f2p = FraunhoferPropagator(focal_grid, pupil_grid, focal_length=focal_length)
    
    # Combine test aberration and defocus phase
    combined_phase = (test_ab + defocus_phase).ravel()
    
    # Create pupil field with combined phase (the electric field at the pupil plane)
    pupil_field = telescope_pupil * np.exp(complex(0, 1) * combined_phase)
    
    # Create a wavefront object
    wavefront = Wavefront(pupil_field, wavelength)
    pupil_image = wavefront.copy()  # Copy of the wavefront, if needed later
    
    # Propagate the wavefront to the focal plane
    focal_field = prop_p2f.forward(wavefront)
    
    # Calculate the focal intensity (taking the magnitude squared of the electric field)
    focal_intensity = np.abs(focal_field.electric_field.reshape(focal_grid.shape))**2
    
    return focal_intensity

def generate_defocus_lists(example_defocus, scales, f, D, test_ab, telescope_pupil, wavelength):
    """
    Generates defocus lists and calculates PSF for each scale.

    Parameters
    ----------
    example_defocus : np.ndarray
        Array representing the example defocus.
    scales : list of float
        List of scales to apply to the defocus phase.
    f : float
        Focal length of the system.
    D : float
        Size of the pupil.
    test_ab : np.ndarray
        Array representing the test aberration.
    telescope_pupil : np.ndarray
        Array representing the telescope pupil.
    wavelength : float
        Wavelength of light in meters.

    Returns
    -------
    tuple
        List of PSFs and list of defocus distances.
    """
    psf_list = []
    distance_list = []
    
    # Define dx_list to match the number of defocus positions (scales)
    #dx_list = [2.0071] * len(scales)  # You can modify this value if needed

    # Ensure the shapes are compatible
    example_defocus = influence_functions[3].shaped
    test_ab = test_ab.reshape(telescope_pupil.shape)

    # Calculate no-defocus image
    no_defocus_phase = np.zeros_like(example_defocus)
    no_defocus_image = propagate_image(no_defocus_phase, test_ab, telescope_pupil, wavelength)
    psf_list.append(no_defocus_image)  # The no-defocus PSF is the first entry

    # Loop over the defocus scales
    for scale in scales:
        # Calculate P2V error and defocus distance
        p2v_radians, delta = calculate_defocus_params(example_defocus, scale, f, D)
        print(f'Scale {scale}: P2V error: {p2v_radians} rad, {p2v_radians/(2*np.pi)} waves, defocus distance: {delta*1e6} microns')
        defocus_image = propagate_image(example_defocus * scale, test_ab, telescope_pupil, wavelength)
        # Generate the defocused image using the scale
        defocus_image = propagate_image(example_defocus * scale, test_ab, telescope_pupil, wavelength)
        psf_list.append(defocus_image)  # Append the defocused PSF to the list
        distance_list.append(delta * 1e6)  # Append the defocus distance in microns

    # Return the PSF list, distance list, and dx list
    return psf_list, distance_list, dx_list
def defocus_images(psf_list):
    for i, psf in enumerate(psf_list):
        plt.figure()
        # using Jules eq
        plt.imshow(np.log10(psf / psf.max()), vmin=-5)
        plt.colorbar()
        plt.title(f'Focal Image {i}')
        plt.show()


In [19]:

# Unified Function

def run_focus_diverse_phase_retrieval_v2(scales=[2, 1], test_ab_scale=0.75):
    """
    This function performs focus diverse phase retrieval, including wavefront propagation,
    defocus handling via Zernike polynomials, and error calculation using a masking pupil,
    ensuring that the grid and array sizes are correct throughout.
    
    Parameters
    ----------
    scales : list of float
        List of defocus scales to apply.
    test_ab_scale : float
        Scaling factor for the test aberration.
    
    Returns
    -------
    dict
        Contains key outputs like pupil phase, difference image, PSF list, P2V error, RMS error, and median error in nm.
    """
    
    # 1. Set up pupil size, focal length, and pupil grid
    pupil_grid = make_pupil_grid(grid_size, pupil_size)
    aperture = make_circular_aperture(pupil_size)
    telescope_pupil = aperture(pupil_grid)
    small_aperture = make_circular_aperture(small_pupil_size)
    masking_pupil = small_aperture(pupil_grid)

    # Ensure `telescope_pupil` and `masking_pupil` are reshaped to match 2D grid size
    pupil_phase_shape = (grid_size, grid_size)  # Assuming the grid is square
    telescope_pupil = telescope_pupil.reshape(pupil_phase_shape)
    masking_pupil = masking_pupil.reshape(pupil_phase_shape)

    # 2. Build wavefront and focal grid
    wavefront = Wavefront(telescope_pupil, wavelength=wavelength)
    focal_grid = make_focal_grid(q=q, num_airy=num_airy, pupil_diameter=pupil_size, focal_length=focal_length, reference_wavelength=wavelength)
    
    # 3. Define the propagators
    prop_p2f = FraunhoferPropagator(pupil_grid, focal_grid, focal_length=focal_length)
    prop_f2p = FraunhoferPropagator(focal_grid, pupil_grid, focal_length=focal_length)
    
    # 4. Save a copy of the original wavefront and focal plane image
    pupil_image = wavefront.copy()
    focal_image = prop_p2f.forward(wavefront)
    perfect_focal = focal_image.copy()
    
    # 5. Generate Zernike basis and handle defocus mode
    influence_functions = make_zernike_basis(256, pupil_size, pupil_grid)#x could be grid_size
    test_ab = test_ab_scale * influence_functions[6]  # Coma
    example_defocus = influence_functions[3].reshape(telescope_pupil.shape)   # Defocus
    
    # 6. Generate defocus lists and calculate PSFs
    psf_list, distance_list, dx_list = generate_defocus_lists(example_defocus, scales, f, D, test_ab, telescope_pupil, wavelength)
    
    # 7. Perform phase diversity retrieval & Phase retrieval loop
    mp = FocusDiversePhaseRetrieval(psf_list, wavelength, dx_list, distance_list)
    test_phase = 0 
    for i in range(200):
        psf00 = mp.step()
    
    # Output intensity and pupil phase
    plt.imshow(np.angle(psf00))
    plt.colorbar()
    
    # Special instrument configuration for SEAL
    seal_params = {
        'image_dx': 2.0071,
        'efl': focal_length * 1e3,
        'wavelength': 0.65,  # Ensure units are correct (microns)
        'pupil_size': pupil_size * 1e3
    }
    conf = InstrumentConfiguration(seal_params)
    
    # 8. Inverse Fourier Transform to get the pupil phase
    raw_pupil_phase = np.angle(mft_rev(psf00, conf))
    
    # 9. Resize the image to our desired output and apply masking pupil
    pupil_phase = resize(raw_pupil_phase, (256, 256))*telescope_pupil.shaped
    #pupil_phase = resize(raw_pupil_phase, pupil_phase_shape) * telescope_pupil  # Ensure proper resizing
    plt.imshow(pupil_phase)
    plt.colorbar()
    print(f'P2V error of resized pupil phase: {np.max(pupil_phase) - np.min(pupil_phase)}')
    
    #Know what out original error is
    pupil_image.electric_field = np.exp(complex(0, 1)*telescope_pupil*(test_ab))
    print(f'P2V error of Original injected error: {np.max(pupil_image.phase.shaped) - np.min(pupil_image.phase.shaped)}')
    
    # Subtract median using masking pupil
    med_subtracted = pupil_phase - np.median(pupil_phase[masking_pupil > 0])
    # Ensure consistent reshaping for `pupil_image.phase` before subtraction
    pupil_image_phase = pupil_image.phase.reshape(pupil_phase_shape)  # Reshape to 2D grid
    
    # Calculate the difference between the pupil image phase and the median-subtracted phase
    difference_image = pupil_image_phase - med_subtracted
    plt.imshow(difference_image * masking_pupil)
    plt.colorbar()
        
    # 10. Calculate error in the masked region
    check_error_region = (pupil_image_phase - med_subtracted)[masking_pupil > 0]
    print(f'Median error of {np.median(check_error_region)} radians.') 
    nm_med = phase_to_m(np.median(check_error_region), wavelength) * 1e9
    print(f'Median error in nanometers: {nm_med} nm')
    # 11. RMS Error Calculation
    print(f'Check Error Region: {check_error_region}')
    print(f'Median of Check Error Region: {np.median(check_error_region)}')
    valid_phase_values = med_subtracted[telescope_pupil > 0]
    mean_phase = np.mean(valid_phase_values)
    rms_error = np.sqrt(np.mean((valid_phase_values - mean_phase) ** 2))
    print(f"RMS error: {rms_error} radians")
    
    # Return relevant results
    return {
        "pupil_phase": pupil_phase,
        "difference_image": difference_image,
        "psf_list": psf_list,
        "p2v_error": np.max(pupil_phase) - np.min(pupil_phase),
        "rms_error": rms_error,
        "nm_med": nm_med
    }


In [20]:

# Example scales and test_ab_scale values
scales = [2, 1]  
test_ab_scale = 0.5 

# Run the simulation
results = run_focus_diverse_phase_retrieval_v2(scales, test_ab_scale)

# Print the key outputs
#print("Pupil Phase:", results["pupil_phase"])
#print("Difference Image:", results["difference_image"])
#print("P2V Error:", results["p2v_error"])
#print("RMS Error:", results["rms_error"])
#print("Median Error in nm:", results["nm_med"])

# If you want to visualize the difference image, you can do:
import matplotlib.pyplot as plt
plt.imshow(results["difference_image"])
plt.colorbar()
plt.title("Difference Image")
plt.show()

NameError: name 'influence_functions' is not defined

In [ ]:

check_med= ma.shape(med_subtracted)
print(check_med)
check_differece= ma.shape(difference_image)
print(check_differece)
check_pupil_image= ma.shape(pupil_image.phase.shaped)
print(check_pupil_image)
check_masking_pupil= ma.shape(masking_pupil.shaped)
print(check_masking_pupil)